In [ ]:
import pyspedas
from pytplot import get_data
import pandas as pd
import numpy as np
from datetime import timedelta
import os

# === USER INPUT ===
t1 = '2016-03-06/00:24:00'  # Start time
t2 = '2016-03-06/00:34:00'  # End time
probes = [1]  # MMS probe(s)

# === TIME FORMATTING ===
t1_ext = (pd.to_datetime(t1) - timedelta(minutes=5)).strftime('%Y-%m-%d/%H:%M:%S')
t2_ext = (pd.to_datetime(t2) + timedelta(minutes=5)).strftime('%Y-%m-%d/%H:%M:%S')
t1_date = t1.split("/")[0]
t1_fmt = t1.split("/")[1].replace(":", "_")
t2_fmt = t2.split("/")[1].replace(":", "_")
folder_name = f'data_{t1_date}_{t1_fmt}_to_{t2_fmt}'
os.makedirs(folder_name, exist_ok=True)

t1_aware = pd.to_datetime(t1).tz_localize('UTC')
t2_aware = pd.to_datetime(t2).tz_localize('UTC')

# === UTILITY FUNCTIONS ===
def convert_to_datetime(time_data):
    return pd.to_datetime(time_data, unit='ns', utc=True)

def interpolate_to_common_time(original_time, original_data, common_time):
    common_sec = common_time.astype(int) / 1e9
    original_sec = original_time.astype(int) / 1e9
    if original_data.ndim == 1:
        return np.interp(common_sec, original_sec, original_data)
    return np.array([
        np.interp(common_sec, original_sec, original_data[:, i])
        for i in range(original_data.shape[1])
    ]).T

# === MAIN LOOP ===
for probe in probes:
    # Load all required data
    pyspedas.mms.fgm(probe=probe, trange=[t1_ext, t2_ext], time_clip=True)
    pyspedas.mms.fpi(probe=probe, datatype=['des-moms'], center_measurement=True, trange=[t1_ext, t2_ext], time_clip=True)
    pyspedas.mms.mec(probe=probe, trange=[t1_ext, t2_ext], time_clip=True)
    pyspedas.mms.dsp(probe=probe, trange=[t1_ext, t2_ext], datatype=['epsd'], data_rate='fast', level='l2', time_clip=True)

    # Get magnetic and plasma data
    B = get_data(f'mms{probe}_fgm_b_gse_srvy_l2_btot', xarray=True)
    B_vec = get_data(f'mms{probe}_fgm_b_gse_srvy_l2', xarray=True)
    density = get_data(f'mms{probe}_des_numberdensity_fast', xarray=True)
    pos = get_data(f'mms{probe}_mec_r_gse', xarray=True)
    mlat = get_data(f'mms{probe}_mec_mlat', xarray=True)
    l_dipole = get_data(f'mms{probe}_mec_l_dipole', xarray=True)
    mlt = get_data(f'mms{probe}_mec_mlt', xarray=True)
    dst = get_data(f'mms{probe}_mec_dst', xarray=True)
    kp = get_data(f'mms{probe}_mec_kp', xarray=True)
    epsd = get_data(f'mms{probe}_dsp_epsd_x', xarray=True)

    # Convert main time axis
    common_time = convert_to_datetime(B.time.data)

    # Interpolate all other datasets to common_time
    B_vec_interp = interpolate_to_common_time(convert_to_datetime(B_vec.time.data), B_vec.data, common_time)
    density_interp = interpolate_to_common_time(convert_to_datetime(density.time.data), density.data, common_time)
    pos_interp = interpolate_to_common_time(convert_to_datetime(pos.time.data), pos.data, common_time)
    mlat_interp = interpolate_to_common_time(convert_to_datetime(mlat.time.data), mlat.data, common_time)
    l_dipole_interp = interpolate_to_common_time(convert_to_datetime(l_dipole.time.data), l_dipole.data, common_time)
    mlt_interp = interpolate_to_common_time(convert_to_datetime(mlt.time.data), mlt.data, common_time)
    dst_interp = interpolate_to_common_time(convert_to_datetime(dst.time.data), dst.data, common_time)
    kp_interp = interpolate_to_common_time(convert_to_datetime(kp.time.data), kp.data, common_time)

    # === EPSD interpolation ===
    if epsd is not None:
        epsd_time = convert_to_datetime(epsd.time.data)
        epsd_freq = epsd.v.data
        epsd_power = epsd.data  # shape: [time, freq]
        epsd_interp = np.empty((len(common_time), len(epsd_freq)))
        for i in range(len(epsd_freq)):
            epsd_interp[:, i] = np.interp(
                common_time.astype(int) / 1e9,
                epsd_time.astype(int) / 1e9,
                epsd_power[:, i]
            )
        # Create frequency column names
        epsd_cols = [f'{f:.1f}Hz' for f in epsd_freq]
        # Filter to 50–800 Hz
        selected_cols = [col for col in epsd_cols if 50.0 <= float(col.replace('Hz', '')) <= 800.0]
        selected_indices = [epsd_cols.index(col) for col in selected_cols]
        epsd_interp = epsd_interp[:, selected_indices]
    else:
        selected_cols = []
        epsd_interp = None
        print(f"No EPSD data for MMS{probe}")

    # === Filter to t1–t2 ===
    mask = (common_time >= t1_aware) & (common_time <= t2_aware)
    filtered_time = common_time[mask]
    rel_time_sec = (filtered_time - filtered_time[0]).total_seconds()

    # === Build DataFrame ===
    df = pd.DataFrame({
        'Time': filtered_time.strftime('%Y-%m-%d/%H:%M:%S.%f'),
        'RelativeTime': rel_time_sec,
        'Bt': B.data[mask],
        'Bx': B_vec_interp[mask, 0],
        'By': B_vec_interp[mask, 1],
        'Bz': B_vec_interp[mask, 2],
        'Density': density_interp[mask],
        'Rx': pos_interp[mask, 0],
        'Ry': pos_interp[mask, 1],
        'Rz': pos_interp[mask, 2],
        'L_Dipole': l_dipole_interp[mask],
        'MLT': mlt_interp[mask],
        'MLat': mlat_interp[mask],
        'Dst': dst_interp[mask],
        'Kp': kp_interp[mask]
    })

    # Add EPSD columns if available
    if epsd_interp is not None:
        for idx, col in enumerate(selected_cols):
            df[col] = epsd_interp[mask, idx]

    # === Save CSV ===
    out_file = f'{folder_name}/mms{probe}_allvars_epsd_{t1_date}_{t1_fmt}_to_{t2_fmt}.csv'
    df.to_csv(out_file, index=False)
    print(f"Data with EPSD saved to {out_file}")


In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import find_peaks

# Load dataset
file_path = "data_2016-03-06_00_24_00_to_00_26_00/mms1_multidimensional_2016-03-06_00_24_00_to_00_26_00.nc"
ds = xr.open_dataset(file_path)
bt = ds['Bt'].values
epsd = ds['EPSD'].values
time = ds['time'].values
freq = ds['frequency'].values

# Find peaks in Bt (significant fluctuations)
peaks, _ = find_peaks(bt, distance=20, prominence=0.5)

# Use Gaussian envelopes around each peak
window_width = 30  # number of points (broader = smoother)

# Gaussian kernel for smooth weighting
def gaussian_kernel(center, width, size):
    x = np.arange(size)
    return np.exp(-0.5 * ((x - center) / width)**2)

# Initialize weighted EPSD
epsd_weighted = np.zeros_like(epsd)

for peak in peaks:
    gk = gaussian_kernel(peak, window_width, len(bt))
    for i in range(len(freq)):
        epsd_weighted[:, i] += epsd[:, i] * gk

# Optional: Normalize per frequency to enhance contrast
epsd_weighted /= np.nanmax(epsd_weighted, axis=0, keepdims=True)

# === Plot ===
fig, ax1 = plt.subplots(figsize=(12, 6))

pcm = ax1.pcolormesh(time, freq, epsd_weighted.T, shading='auto', cmap='jet')
ax1.set_yscale('log')
ax1.set_ylabel("Frequency (Hz)")
fig.colorbar(pcm, ax=ax1, label='EPSD (Gaussian-weighted)')

# Overlay Bt
ax2 = ax1.twinx()
ax2.plot(time, bt, color='white', linewidth=1.5, label='Bt (nT)')
ax2.plot(time[peaks], bt[peaks], "o", color='cyan', label='Bt Peaks')
ax2.set_ylabel("Bt (nT)", color='white')
ax2.tick_params(axis='y', labelcolor='white')
ax2.spines['right'].set_color('white')

ax1.set_title("EPSD Weighted by Bt Fluctuation Envelope")
plt.tight_layout()
plt.show()


In [ ]:
import pyspedas
from pytplot import get_data
import pandas as pd
import numpy as np
import xarray as xr
from datetime import timedelta
import os

# === USER INPUT ===
t1 = '2016-03-06/00:24:00'
t2 = '2016-03-06/00:34:00'
probes = [1]

# === TIME FORMATTING ===
t1_ext = (pd.to_datetime(t1) - timedelta(minutes=5)).strftime('%Y-%m-%d/%H:%M:%S')
t2_ext = (pd.to_datetime(t2) + timedelta(minutes=5)).strftime('%Y-%m-%d/%H:%M:%S')
t1_date = t1.split("/")[0]
t1_fmt = t1.split("/")[1].replace(":", "_")
t2_fmt = t2.split("/")[1].replace(":", "_")
folder_name = f'data_{t1_date}_{t1_fmt}_to_{t2_fmt}'
os.makedirs(folder_name, exist_ok=True)

t1_aware = pd.to_datetime(t1).tz_localize('UTC')
t2_aware = pd.to_datetime(t2).tz_localize('UTC')

def convert_to_datetime(time_data):
    return pd.to_datetime(time_data, unit='ns', utc=True)

def interpolate_to_common_time(original_time, original_data, common_time):
    common_sec = common_time.astype(int) / 1e9
    original_sec = original_time.astype(int) / 1e9
    if original_data.ndim == 1:
        return np.interp(common_sec, original_sec, original_data)
    return np.array([
        np.interp(common_sec, original_sec, original_data[:, i])
        for i in range(original_data.shape[1])
    ]).T

# === MAIN LOOP ===
for probe in probes:
    # Load data
    pyspedas.mms.fgm(probe=probe, trange=[t1_ext, t2_ext], time_clip=True)
    pyspedas.mms.fpi(probe=probe, datatype=['des-moms'], center_measurement=True, trange=[t1_ext, t2_ext], time_clip=True)
    pyspedas.mms.mec(probe=probe, trange=[t1_ext, t2_ext], time_clip=True)
    pyspedas.mms.dsp(probe=probe, trange=[t1_ext, t2_ext], datatype=['epsd'], data_rate='fast', level='l2', time_clip=True)

    # Get data
    B = get_data(f'mms{probe}_fgm_b_gse_srvy_l2_btot', xarray=True)
    B_vec = get_data(f'mms{probe}_fgm_b_gse_srvy_l2', xarray=True)
    density = get_data(f'mms{probe}_des_numberdensity_fast', xarray=True)
    pos = get_data(f'mms{probe}_mec_r_gse', xarray=True)
    mlat = get_data(f'mms{probe}_mec_mlat', xarray=True)
    l_dipole = get_data(f'mms{probe}_mec_l_dipole', xarray=True)
    mlt = get_data(f'mms{probe}_mec_mlt', xarray=True)
    dst = get_data(f'mms{probe}_mec_dst', xarray=True)
    kp = get_data(f'mms{probe}_mec_kp', xarray=True)
    epsd = get_data(f'mms{probe}_dsp_epsd_x', xarray=True)

    # Time base
    common_time = convert_to_datetime(B.time.data)

    # Interpolate to common time
    B_vec_interp = interpolate_to_common_time(convert_to_datetime(B_vec.time.data), B_vec.data, common_time)
    density_interp = interpolate_to_common_time(convert_to_datetime(density.time.data), density.data, common_time)
    pos_interp = interpolate_to_common_time(convert_to_datetime(pos.time.data), pos.data, common_time)
    mlat_interp = interpolate_to_common_time(convert_to_datetime(mlat.time.data), mlat.data, common_time)
    l_dipole_interp = interpolate_to_common_time(convert_to_datetime(l_dipole.time.data), l_dipole.data, common_time)
    mlt_interp = interpolate_to_common_time(convert_to_datetime(mlt.time.data), mlt.data, common_time)
    dst_interp = interpolate_to_common_time(convert_to_datetime(dst.time.data), dst.data, common_time)
    kp_interp = interpolate_to_common_time(convert_to_datetime(kp.time.data), kp.data, common_time)

    # EPSD interpolation
    if epsd is not None:
        epsd_time = convert_to_datetime(epsd.time.data)
        epsd_freq = epsd.v.data
        epsd_power = epsd.data
        epsd_interp = np.empty((len(common_time), len(epsd_freq)))
        for i in range(len(epsd_freq)):
            epsd_interp[:, i] = np.interp(
                common_time.astype(int) / 1e9,
                epsd_time.astype(int) / 1e9,
                epsd_power[:, i]
            )
        # Filter to 50–800 Hz
        freq_mask = (epsd_freq >= 50.0) & (epsd_freq <= 800.0)
        epsd_freq = epsd_freq[freq_mask]
        epsd_interp = epsd_interp[:, freq_mask]
    else:
        print(f"No EPSD data for MMS{probe}")
        epsd_freq = np.array([])
        epsd_interp = np.empty((len(common_time), 0))

    # Filter time window
    mask = (common_time >= t1_aware) & (common_time <= t2_aware)
    filtered_time = common_time[mask]
    rel_time_sec = (filtered_time - filtered_time[0]).total_seconds()

    # Create xarray Dataset
    ds = xr.Dataset(
        data_vars=dict(
            Bt=("time", B.data[mask]),
            Bx=("time", B_vec_interp[mask, 0]),
            By=("time", B_vec_interp[mask, 1]),
            Bz=("time", B_vec_interp[mask, 2]),
            Density=("time", density_interp[mask]),
            Rx=("time", pos_interp[mask, 0]),
            Ry=("time", pos_interp[mask, 1]),
            Rz=("time", pos_interp[mask, 2]),
            L_Dipole=("time", l_dipole_interp[mask]),
            MLT=("time", mlt_interp[mask]),
            MLat=("time", mlat_interp[mask]),
            Dst=("time", dst_interp[mask]),
            Kp=("time", kp_interp[mask]),
            EPSD=(["time", "frequency"], epsd_interp[mask])
        ),
        coords=dict(
            time=("time", filtered_time),
            frequency=("frequency", epsd_freq),
            RelativeTime=("time", rel_time_sec)
        ),
        attrs=dict(
            description=f"MMS{probe} full dataset including EPSD",
            source="NASA MMS via pySPEDAS",
        )
    )

    # Save to NetCDF
    out_file = f"{folder_name}/mms{probe}_multidimensional_{t1_date}_{t1_fmt}_to_{t2_fmt}.nc"
    ds.to_netcdf(out_file)
    print(f"Multidimensional dataset saved to {out_file}")



In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import xarray as xr
import numpy as np
import pandas as pd

# Reopen the saved NetCDF file
nc_file = f"{folder_name}/mms{probe}_multidimensional_{t1_date}_{t1_fmt}_to_{t2_fmt}.nc"
ds = xr.open_dataset(nc_file, decode_times=False)

# Convert raw time values (assumed in seconds since epoch) to datetime
raw_time = ds['time'].values

# Attempt conversion assuming time is in seconds since UNIX epoch (1970-01-01)
try:
    time_dt = pd.to_datetime(raw_time, unit='s')
except Exception:
    # Try nanoseconds (some datasets store it this way)
    time_dt = pd.to_datetime(raw_time)

frequency = ds['frequency'].values
epsd = ds['EPSD'].values
bt = ds['Bt'].values

# Create EPSD color plot with Bt overlay
fig, ax1 = plt.subplots(figsize=(12, 6))
pcm = ax1.pcolormesh(time_dt, frequency, epsd.T, shading='auto', cmap='jet')
#ax1.set_yscale('log')
ax1.set_ylabel('Frequency (Hz)')
fig.colorbar(pcm, ax=ax1, label='EPSD Power')

# Format x-axis
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax1.xaxis.set_major_locator(mdates.AutoDateLocator())
ax1.set_xlabel('Time (UTC)')

# Overlay Bt on right axis
ax2 = ax1.twinx()
ax2.plot(time_dt, bt, color='white', linewidth=2, label='Bt (nT)')
ax2.set_ylabel('Bt (nT)', color='black')
ax2.tick_params(axis='y', labelcolor='black')
ax2.spines['right'].set_color('black')
ax2.set_ylim(bt.min() - 5, bt.max() + 5)  # Adjust as needed

ax1.set_title(f"MMS{probe} EPSD and Bt vs Time")

plt.tight_layout()
plt.show()


In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
from scipy.signal import find_peaks

# Load dataset without decoding time automatically
nc_file = f"{folder_name}/mms{probe}_multidimensional_{t1_date}_{t1_fmt}_to_{t2_fmt}.nc"
ds = xr.open_dataset(nc_file, decode_times=False)

bt = ds['Bt'].values
epsd = ds['EPSD'].values
freq = ds['frequency'].values

# Handle raw time values (attempt seconds since epoch, then fallback)
raw_time = ds['time'].values
try:
    time_dt = pd.to_datetime(raw_time, unit='s')
except Exception:
    time_dt = pd.to_datetime(raw_time)

# Find peaks in Bt (significant fluctuations)
peaks, _ = find_peaks(bt, distance=20, prominence=0.5)

# Use Gaussian envelopes around each peak
window_width = 30  # number of points (broader = smoother)

# Gaussian kernel for smooth weighting
def gaussian_kernel(center, width, size):
    x = np.arange(size)
    return np.exp(-0.5 * ((x - center) / width)**2)

# Initialize weighted EPSD
epsd_weighted = np.zeros_like(epsd)

for peak in peaks:
    gk = gaussian_kernel(peak, window_width, len(bt))
    for i in range(len(freq)):
        epsd_weighted[:, i] += epsd[:, i] * gk

# Normalize per frequency
epsd_weighted /= np.nanmax(epsd_weighted, axis=0, keepdims=True)

# === Plot ===
fig, ax1 = plt.subplots(figsize=(12, 6))

pcm = ax1.pcolormesh(time_dt, freq, epsd_weighted.T, shading='auto', cmap='jet')
ax1.set_yscale('log')
ax1.set_ylabel("Frequency (Hz)")
fig.colorbar(pcm, ax=ax1, label='EPSD (Gaussian-weighted)')

# Format x-axis as datetime
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax1.xaxis.set_major_locator(mdates.AutoDateLocator())
ax1.set_xlabel('Time (UTC)')

# Overlay Bt
ax2 = ax1.twinx()
ax2.plot(time_dt, bt, color='white', linewidth=2, label='Bt (nT)')
ax2.set_ylabel('Bt (nT)', color='black')
ax2.tick_params(axis='y', labelcolor='black')
ax2.spines['right'].set_color('black')
ax2.set_ylim(bt.min() - 5, bt.max() + 5)  # Adjust as needed
ax2.spines['right'].set_color('white')

ax1.set_title("EPSD Weighted by Bt Fluctuation Envelope")
plt.tight_layout()
plt.show()


# Model Training
## RandomForestClassifier

In [ ]:

import os
import numpy as np
import pandas as pd
import xarray as xr
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

SAMPLE_FOLDER = "80_samples"
DATA_FOLDER = "DATA_week"
MODEL_PATH = "ducting_detector_model.pkl"
SCALER_PATH = "scaler_RandomForestClassifier1.pkl"
WINDOW_SIZE = 160
STEP_SIZE = 80

def parse_filename_time(filename):
    try:
        name = filename.replace(".nc", "")
        parts = name.split('_')
        date_str = parts[2]
        start_hms = [int(parts[3]), int(parts[4]), int(parts[5])]
        end_hms   = [int(parts[7]), int(parts[8]), int(parts[9])]
        start_time = pd.Timestamp(date_str) + pd.Timedelta(
            hours=start_hms[0], minutes=start_hms[1], seconds=start_hms[2])
        end_time = pd.Timestamp(date_str) + pd.Timedelta(
            hours=end_hms[0], minutes=end_hms[1], seconds=end_hms[2])
        return start_time, end_time
    except Exception as e:
        print(f"Filename parsing failed for {filename}: {e}")
        return None, None

def extract_features(ds):
    base = pd.DataFrame({
        "Bt": ds["Bt"].values,
        "Density": ds["Density"].values,
        "MLT": ds["MLT"].values,
        "MLat": ds["MLat"].values,
        "Dst": ds["Dst"].values,
        "Kp": ds["Kp"].values,
    })
    epsd = ds["EPSD"].values
    for i in range(epsd.shape[1]):
        base[f"EPSD_f{i}"] = epsd[:, i]
    return base

def get_sample_intervals():
    intervals = []
    for file in os.listdir(SAMPLE_FOLDER):
        if file.endswith(".nc"):
            start, end = parse_filename_time(file)
            if start and end:
                intervals.append((start, end))
    print(f"✅ Found {len(intervals)} labeled sample intervals.")
    return intervals

def label_window(window_start, window_duration=10, intervals=None):
    window_end = window_start + pd.Timedelta(seconds=window_duration)
    for (istart, iend) in intervals:
        if (istart <= window_end and iend >= window_start):
            return 1
    return 0

# === TRAINING STARTS HERE ===

all_features = []
all_labels = []

sample_intervals = get_sample_intervals()
print("Sample intervals preview:")
for s in sample_intervals[:3]:
    print(" -", s)

for file in os.listdir(DATA_FOLDER):
    if not file.endswith(".nc"):
        continue

    file_path = os.path.join(DATA_FOLDER, file)
    ds = xr.open_dataset(file_path, decode_times=True)

    try:
        times = pd.to_datetime(ds["time"].values)
    except Exception as e:
        print(f"⛔️ Failed to extract time from {file}: {e}")
        continue

    features = extract_features(ds)

    for i in range(0, len(features) - WINDOW_SIZE, STEP_SIZE):
        window = features.iloc[i:i + WINDOW_SIZE]
        if len(window) < WINDOW_SIZE:
            continue

        flat = window.mean().values
        if np.isnan(flat).any() or flat.ndim != 1:
            continue

        window_time = times[i]
        label = label_window(window_time, intervals=sample_intervals)

        all_features.append(flat)
        all_labels.append(label)

# === FINAL PREP ===

X = np.array(all_features)
y = np.array(all_labels)

print("✅ Feature shape:", X.shape)
if X.ndim != 2:
    raise ValueError("X is not 2D. Check feature extraction or window flattening.")

valid_indices = ~np.isnan(X).any(axis=1)
X = X[valid_indices]
y = y[valid_indices]

print("✅ Final label distribution:", np.bincount(y))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y if len(np.unique(y)) > 1 else None
)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print("\n🧠 Model Evaluation:")
print(classification_report(y_test, y_pred))

joblib.dump(clf, MODEL_PATH)
joblib.dump(scaler, SCALER_PATH)
print("✅ Model and scaler saved successfully.")


In [ ]:
print("Label distribution:", np.bincount(y))


In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import joblib
from scipy.signal import find_peaks
from sklearn.preprocessing import StandardScaler

MODEL_PATH = "ducting_detector_model.pkl"
SCALER_PATH = "scaler_RandomForestClassifier1.pkl"
WINDOW_SIZE = 100
STEP_SIZE = 80
THRESHOLD = 0.35  # adjust as needed
OUTPUT_FOLDER = "predicted_events_RandomForestClassifier1"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

def extract_features(ds):
    base = pd.DataFrame({
        "Bt": ds["Bt"].values,
        "Density": ds["Density"].values,
        "MLT": ds["MLT"].values,
        "MLat": ds["MLat"].values,
        "Dst": ds["Dst"].values,
        "Kp": ds["Kp"].values,
    })
    epsd = ds["EPSD"].values
    for i in range(epsd.shape[1]):
        base[f"EPSD_f{i}"] = epsd[:, i]

    # Optional: interpolate or drop NaNs
    base = base.interpolate(limit_direction='both')  # or base.dropna()
    return base


def predict_events(ds, features, times, model, scaler, threshold=THRESHOLD):
    events = []
    for i in range(0, len(features) - WINDOW_SIZE, STEP_SIZE):
        window = features.iloc[i:i + WINDOW_SIZE]
        flat = window.mean().values.reshape(1, -1)

        # ⛔️ Skip if NaNs are found
        if np.isnan(flat).any():
            continue

        flat_scaled = scaler.transform(flat)
        prob = model.predict_proba(flat_scaled)[0][1]
        print(f"Window {i}: P(event) = {prob:.2f}")
        if prob >= threshold:
            start_time = times[i]
            end_time = times[i + WINDOW_SIZE - 1]
            events.append((i, start_time, end_time))
    return events


def save_event(ds, i, start_time, end_time, file_base):
    event_ds = ds.isel(time=slice(i, i + WINDOW_SIZE))
    out_name = f"{file_base}_{start_time.strftime('%H_%M_%S')}_to_{end_time.strftime('%H_%M_%S')}.nc"
    out_path = os.path.join(OUTPUT_FOLDER, out_name)
    event_ds.to_netcdf(out_path)
    print(f"✅ Saved event: {out_path}")

def plot_event(ds, i, start_time, end_time):
    bt = ds["Bt"].values
    epsd = ds["EPSD"].values
    freq = ds["frequency"].values
    time = pd.to_datetime(ds["time"].values)

    bt_segment = bt[i:i + WINDOW_SIZE]
    epsd_segment = epsd[i:i + WINDOW_SIZE, :]
    time_segment = time[i:i + WINDOW_SIZE]

    peaks, _ = find_peaks(bt_segment, distance=20, prominence=0.5)

    def gaussian_kernel(center, width, size):
        x = np.arange(size)
        return np.exp(-0.5 * ((x - center) / width) ** 2)

    epsd_weighted = np.zeros_like(epsd_segment)
    for peak in peaks:
        gk = gaussian_kernel(peak, 30, len(bt_segment))
        for f in range(len(freq)):
            epsd_weighted[:, f] += epsd_segment[:, f] * gk
    epsd_weighted /= np.nanmax(epsd_weighted, axis=0, keepdims=True)

    fig, ax1 = plt.subplots(figsize=(12, 6))
    pcm = ax1.pcolormesh(time_segment, freq, epsd_weighted.T, shading='auto', cmap='jet')
    ax1.set_yscale('log')
    ax1.set_ylabel("Frequency (Hz)")
    fig.colorbar(pcm, ax=ax1, label='EPSD (Gaussian-weighted)')

    ax2 = ax1.twinx()
    ax2.plot(time_segment, bt_segment, color='white', linewidth=1.5)
    ax2.plot(time_segment[peaks], bt_segment[peaks], "o", color='cyan')
    ax2.set_ylabel("Bt (nT)", color='white')
    ax2.tick_params(axis='y', labelcolor='white')
    ax2.spines['right'].set_color('white')

    ax1.set_title(f"Predicted Ducting Event: {start_time} to {end_time}")
    plt.tight_layout()
    jpg_name = f"{start_time.strftime('%Y-%m-%d_%H-%M-%S')}_to_{end_time.strftime('%H-%M-%S')}.jpg"
    jpg_path = os.path.join(OUTPUT_FOLDER, jpg_name)
    plt.savefig(jpg_path, dpi=300)
    plt.close()
    print(f"🖼️ Saved plot: {jpg_path}")

def plot_event(ds, i, start_time, end_time):
    bt = ds["Bt"].values
    epsd = ds["EPSD"].values
    freq = ds["frequency"].values
    time = pd.to_datetime(ds["time"].values)

    bt_segment = bt[i:i + WINDOW_SIZE]
    epsd_segment = epsd[i:i + WINDOW_SIZE, :]
    time_segment = time[i:i + WINDOW_SIZE]

    fig, ax1 = plt.subplots(figsize=(12, 6))
    pcm = ax1.pcolormesh(time_segment, freq, epsd_segment.T, shading='auto', cmap='jet')
    ax1.set_yscale('log')
    ax1.set_ylabel("Frequency (Hz)")
    fig.colorbar(pcm, ax=ax1, label='EPSD')

    # Overlay Bt
    ax2 = ax1.twinx()
    ax2.plot(time, bt, color='white', linewidth=1.5, label='Bt (nT)')
    # Add extrema as dots
    extrema_indices = np.argwhere((bt == np.max(bt)) | (bt == np.min(bt))).flatten()
    ax2.plot(time[extrema_indices], bt[extrema_indices], "o", color='cyan', label='Bt Extrema')

    ax2.set_ylabel("Bt (nT)", color='white')
    ax2.tick_params(axis='y', labelcolor='white')
    ax2.spines['right'].set_color('white')

    ax1.set_title(f"Predicted Ducting Event: {start_time} to {end_time}")
    plt.tight_layout()

    jpg_name = f"{start_time.strftime('%Y-%m-%d_%H-%M-%S')}_to_{end_time.strftime('%H-%M-%S')}.jpg"
    jpg_path = os.path.join(OUTPUT_FOLDER, jpg_name)
    plt.savefig(jpg_path, dpi=300)
    plt.close()
    print(f"🖼️ Saved plot: {jpg_path}")


def run_detection(file_path):
    print(f"\n🚀 Predicting for: {file_path}")
    model = joblib.load(MODEL_PATH)
    scaler = joblib.load(SCALER_PATH)

    ds = xr.open_dataset(file_path, decode_times=True)
    times = pd.to_datetime(ds["time"].values)
    features = extract_features(ds)

    events = predict_events(ds, features, times, model, scaler)
    print(f"🧠 Detected {len(events)} events.")

    file_base = os.path.basename(file_path).replace(".nc", "")
    for i, start, end in events:
        save_event(ds, i, start, end, file_base)
        plot_event(ds, i, start, end)

# === RUN HERE ===
# Example:
run_detection("DATA_week/mms1_multidimensional_2016-03-07_00_00_06_to_23_59_59.nc")


In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import joblib
from sklearn.preprocessing import StandardScaler
from scipy.signal import find_peaks

# === CONFIG ===
MODEL_PATH = "ducting_detector_model.pkl"
SCALER_PATH = "scaler_RandomForestClassifier1.pkl"
WINDOW_SIZE = 80
STEP_SIZE = 20
THRESHOLD = 0.25  # probability threshold to consider a window as a possible event
OUTPUT_FOLDER = "predicted_events_RandomForestClassifier2"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# === FEATURE EXTRACTION ===
def extract_features(ds):
    base = pd.DataFrame({
        "Bt": ds["Bt"].values,
        "Density": ds["Density"].values,
        "MLT": ds["MLT"].values,
        "MLat": ds["MLat"].values,
        "Dst": ds["Dst"].values,
        "Kp": ds["Kp"].values,
    })
    epsd = ds["EPSD"].values
    for i in range(epsd.shape[1]):
        base[f"EPSD_f{i}"] = epsd[:, i]
    return base

# === PREDICTION ===
def predict_probabilities(ds, features, times, model, scaler):
    window_probs = []
    for i in range(0, len(features) - WINDOW_SIZE, STEP_SIZE):
        window = features.iloc[i:i + WINDOW_SIZE]
        flat = window.mean().values.reshape(1, -1)
        if np.isnan(flat).any():
            continue  # Skip windows with NaNs
        flat_scaled = scaler.transform(flat)
        prob = model.predict_proba(flat_scaled)[0][1]
        window_probs.append((i, prob, times[i]))
    return window_probs

# === MERGE DYNAMIC EVENTS ===
def merge_event_windows(prob_windows, threshold):
    events = []
    buffer = []
    for i, prob, _ in prob_windows:
        if prob >= threshold:
            if buffer and i <= buffer[-1][0] + STEP_SIZE:
                buffer.append((i, prob))
            else:
                if buffer:
                    events.append((buffer[0][0], buffer[-1][0] + WINDOW_SIZE))
                buffer = [(i, prob)]
    if buffer:
        events.append((buffer[0][0], buffer[-1][0] + WINDOW_SIZE))
    return events

# === FILE SAVING ===
def save_event(ds, start_idx, end_idx, start_time, end_time, base_name):
    event_ds = ds.isel(time=slice(start_idx, end_idx))
    out_name = f"{base_name}_{start_time.strftime('%H_%M_%S')}_to_{end_time.strftime('%H_%M_%S')}.nc"
    out_path = os.path.join(OUTPUT_FOLDER, out_name)
    event_ds.to_netcdf(out_path)
    print(f"✅ Saved event data: {out_path}")

def plot_event(ds, start_idx, end_idx, start_time, end_time):
    bt = ds["Bt"].values[start_idx:end_idx]
    epsd = ds["EPSD"].values[start_idx:end_idx, :]
    freq = ds["frequency"].values
    time = pd.to_datetime(ds["time"].values[start_idx:end_idx])

    fig, ax1 = plt.subplots(figsize=(12, 6))
    pcm = ax1.pcolormesh(time, freq, epsd.T, shading='auto', cmap='jet')
    ax1.set_yscale('log')
    ax1.set_ylabel("Frequency (Hz)")
    fig.colorbar(pcm, ax=ax1, label='EPSD')

    ax2 = ax1.twinx()
    ax2.plot(time, bt, color='white', linewidth=1.5, label='Bt (nT)')
    # Add extrema as dots
    extrema_indices = np.argwhere((bt == np.max(bt)) | (bt == np.min(bt))).flatten()
    ax2.plot(time[extrema_indices], bt[extrema_indices], "o", color='cyan', label='Bt Extrema')

    ax2.set_ylabel("Bt (nT)", color='white')
    ax2.tick_params(axis='y', labelcolor='white')
    ax2.spines['right'].set_color('white')

    ax1.set_title(f"Predicted Ducting Event: {start_time} to {end_time}")
    plt.tight_layout()
    jpg_name = f"{start_time.strftime('%Y-%m-%d_%H-%M-%S')}_to_{end_time.strftime('%H-%M-%S')}.jpg"
    jpg_path = os.path.join(OUTPUT_FOLDER, jpg_name)
    plt.savefig(jpg_path, dpi=300)
    plt.close()
    print(f"🖼️ Saved plot: {jpg_path}")

# === MAIN DETECTION FUNCTION ===
def run_detection(file_path):
    print(f"\n🚀 Predicting for: {file_path}")
    model = joblib.load(MODEL_PATH)
    scaler = joblib.load(SCALER_PATH)

    ds = xr.open_dataset(file_path, decode_times=True)
    features = extract_features(ds)
    times = pd.to_datetime(ds["time"].values)

    print(f"📅 Time range: {times[0]} to {times[-1]}")
    probs = predict_probabilities(ds, features, times, model, scaler)
    merged_events = merge_event_windows(probs, THRESHOLD)

    print(f"🧠 Detected {len(merged_events)} merged events.")
    base_name = os.path.basename(file_path).replace(".nc", "")
    for start_idx, end_idx in merged_events:
        start_time = times[start_idx]
        end_time = times[min(end_idx, len(times)-1)]
        save_event(ds, start_idx, end_idx, start_time, end_time, base_name)
        plot_event(ds, start_idx, end_idx, start_time, end_time)

# === USAGE ===
# run_detection("DATA_week/your_file_here.nc")


# === USAGE ===
run_detection("DATA_week/mms1_multidimensional_2016-03-07_00_00_06_to_23_59_59.nc")


# Training code for RandomForestClassifier

 Form the sample folder use all the file but the files with 2016-03-07_ in their name for training. Then, Does it use the DATA_week folder to read the mms1_multidimensional_2016-03-07_00_00_06_to_23_59_59.nc file as a test, and then compaare your results with the sample files that include 2016-03-07_  

In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score

# === CONFIG ===
MODEL_PATH = "ducting_detector_model.pkl"
SCALER_PATH = "scaler_RandomForestClassifier1.pkl"
SAMPLE_FOLDER = "All_samples"
DATA_FILE = "DATA_week/mms1_multidimensional_2016-03-07_00_00_06_to_23_59_59.nc"
WINDOW_SIZE = 160
STEP_SIZE = 80
THRESHOLD = 0.35

# === FEATURE EXTRACTION ===
def extract_features(ds):
    base = pd.DataFrame({
        "Bt": ds["Bt"].values,
        "Density": ds["Density"].values,
        "MLT": ds["MLT"].values,
        "MLat": ds["MLat"].values,
        "Dst": ds["Dst"].values,
        "Kp": ds["Kp"].values,
    })
    epsd = ds["EPSD"].values
    for i in range(epsd.shape[1]):
        base[f"EPSD_f{i}"] = epsd[:, i]
    return base

# === PARSE SAMPLE INTERVALS ===
def parse_filename_time(filename):
    name = filename.replace(".nc", "")
    parts = name.split('_')
    date_str = parts[2]
    start_hms = [int(parts[3]), int(parts[4]), int(parts[5])]
    end_hms = [int(parts[7]), int(parts[8]), int(parts[9])]
    start_time = pd.Timestamp(date_str) + pd.Timedelta(
        hours=start_hms[0], minutes=start_hms[1], seconds=start_hms[2])
    end_time = pd.Timestamp(date_str) + pd.Timedelta(
        hours=end_hms[0], minutes=end_hms[1], seconds=end_hms[2])
    return start_time, end_time

def load_ground_truth_intervals():
    intervals = []
    for f in os.listdir(SAMPLE_FOLDER):
        if f.endswith(".nc") and "2016-03-07" in f:
            start, end = parse_filename_time(f)
            intervals.append((start, end))
    print(f"✅ Loaded {len(intervals)} ground truth intervals.")
    return intervals

def predict_event_intervals(model, scaler, ds, features, times):
    probs = []
    for i in range(0, len(features) - WINDOW_SIZE, STEP_SIZE):
        window = features.iloc[i:i + WINDOW_SIZE]
        flat = window.mean().values.reshape(1, -1)
        if np.isnan(flat).any():
            continue
        flat_scaled = scaler.transform(flat)
        prob = model.predict_proba(flat_scaled)[0][1]
        if prob >= THRESHOLD:
            start_time = times[i]
            end_time = times[i + WINDOW_SIZE - 1]
            probs.append((start_time, end_time))
    print(f"✅ Predicted {len(probs)} event intervals.")
    return probs

def intervals_overlap(a_start, a_end, b_start, b_end):
    return not (a_end < b_start or a_start > b_end)

def evaluate(predicted, ground_truth):
    y_true = []
    y_pred = []
    matched = set()
    for gt_start, gt_end in ground_truth:
        found = False
        for i, (pred_start, pred_end) in enumerate(predicted):
            if intervals_overlap(gt_start, gt_end, pred_start, pred_end):
                y_true.append(1)
                y_pred.append(1)
                matched.add(i)
                found = True
                break
        if not found:
            y_true.append(1)
            y_pred.append(0)

    for i, _ in enumerate(predicted):
        if i not in matched:
            y_true.append(0)
            y_pred.append(1)

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    print("\n📈 Evaluation Metrics:")
    print(f"Precision: {precision:.2f}")
    print(f"Recall:    {recall:.2f}")
    print(f"F1 Score:  {f1:.2f}")

# === MAIN ===
print("\n🔍 Getting ground truth intervals...")
gt_intervals = load_ground_truth_intervals()

print("\n🔮 Predicting event intervals...")
model = joblib.load(MODEL_PATH)
scaler = joblib.load(SCALER_PATH)
ds = xr.open_dataset(DATA_FILE, decode_times=True)
times = pd.to_datetime(ds["time"].values)
features = extract_features(ds)
predicted_intervals = predict_event_intervals(model, scaler, ds, features, times)

print("\n📊 Evaluating...")
evaluate(predicted_intervals, gt_intervals)
